# 🟢 ark-gamma-mini-v2 — Sequence-Conditioned GAN

## 🔴 Imports

In [1]:
import os, json, math, random, time, hashlib
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import gc
import psutil
from pathlib import Path
import textwrap

from tokenizers import Tokenizer as HFTokenizer

F:\pyvenv\pyvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ⚙️ Configuration

In [3]:
MODEL_NAME = "ark-gamma-mini-v2"
ROOT = Path.cwd()
if not (ROOT / "dataset").exists() and (ROOT / "ark-vision" / "dataset").exists():
    ROOT = ROOT / "ark-vision"

DATA = ROOT / "dataset" / "Alphabet_VISION_Dataset"

SELECTED_DATASETS = [
    "various_images",
]

TRAIN_ROOT = DATA / "train"

if not TRAIN_ROOT.is_dir():
    raise FileNotFoundError(
        f"Training directory not found: {TRAIN_ROOT}"
    )

if SELECTED_DATASETS is None:
    SELECTED_DATASETS = sorted(
        directory.name
        for directory in TRAIN_ROOT.iterdir()
        if directory.is_dir()
    )

if not SELECTED_DATASETS:
    raise ValueError("No datasets selected.")

TRAIN_DIRS = [
    TRAIN_ROOT / name
    for name in SELECTED_DATASETS
]

for directory in TRAIN_DIRS:
    if not directory.is_dir():
        raise FileNotFoundError(
            f"Dataset directory not found: {directory}"
        )

print("Selected datasets:", SELECTED_DATASETS)

print("\nTraining directories:")
for directory in TRAIN_DIRS:
    print(" -", directory)
    
TOKENIZER_PATH = ROOT / "tokenizer" / "bpe-tokenizer_alphabet_nlp_dataset_29,000.json"
LOG_ROOT = ROOT / "logs" / MODEL_NAME

DATA_DIR = DATA

# Automatic hardware detection
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Automatic image mode selection
IMAGE_MODE = "RGB"
IMAGE_CHANNELS = 3

print(
    f"Device: {DEVICE} | "
    f"Image mode: {IMAGE_MODE} | "
    f"Channels: {IMAGE_CHANNELS}"
)

IMAGE_SIZE = (64, 64)
MAX_TEXT_LENGTH = 64
BATCH_SIZE = 4
EPOCHS = 50
LR_G = 2e-4
LR_D = 1e-4
LATENT_DIM = 64
TEXT_DIM = 128
BASE_CHANNELS = 32
NUM_HEADS = 4
NUM_TEXT_LAYERS = 2
LAMBDA_RECON_SINGLE = 10.0
LAMBDA_MISMATCH = 0.5
LABEL_SMOOTHING = 0.05
RESUME_FROM_PREVIOUS = True
SEED = 42
NUM_WORKERS = 0
LOG_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print(f"Device={DEVICE} | mode={IMAGE_MODE} | image size={IMAGE_SIZE}")
print("Project root:", ROOT)
print("Tokenizer:", TOKENIZER_PATH)
print("Dataset:", DATA_DIR)
assert TOKENIZER_PATH.is_file(), "Tokenizer not found!"
assert DATA_DIR.is_dir(), "Dataset directory not found!"
print("Tokenizer and dataset paths are valid.")

Selected datasets: ['various_images']

Training directories:
 - F:\pyvenv\ARK\ark-vision\dataset\Alphabet_VISION_Dataset\train\various_images
Device: cpu | Image mode: RGB | Channels: 3
Device=cpu | mode=RGB | image size=(64, 64)
Project root: F:\pyvenv\ARK\ark-vision
Tokenizer: F:\pyvenv\ARK\ark-vision\tokenizer\bpe-tokenizer_alphabet_nlp_dataset_29,000.json
Dataset: F:\pyvenv\ARK\ark-vision\dataset\Alphabet_VISION_Dataset
Tokenizer and dataset paths are valid.


## 🗣️ BPE Tokenizer

In [4]:
backend = HFTokenizer.from_file(str(TOKENIZER_PATH))
vocab = backend.get_vocab()

print("Vocabulary size:", backend.get_vocab_size())
print("Configured padding:", backend.padding)
print("Configured post-processor:", backend.post_processor)
print("\nPossible special tokens:")

for token, token_id in sorted(
    vocab.items(), key=lambda item: item[1]
):
    if any(key in token.lower() for key in (
        "pad", "unk", "bos", "eos", "cls", "sep"
    )):
        print(f"{token!r}: {token_id}")

print("\nTokenizer special tokens:")
print(backend.get_added_tokens_decoder())

Vocabulary size: 29873
Configured padding: None
Configured post-processor: None

Possible special tokens:
'|<unk>|': 0
'tepad': 11276
'ĠNotepad': 13527
'cls': 17324
'Boson': 18378
'Sep': 18710
'ĠBost': 23117
'ĠBoson': 23124
'boston': 24860
'ĠSept': 28192
'Ġnotepad': 28246
'ĠJoseph': 28352
'Bosons': 28541
'Dunkle': 28561
'Homeostasis': 28593
'Septemb': 28675
'ĠBoston': 29491
'Dunkleosteus': 29845
'September': 29857

Tokenizer special tokens:
{0: AddedToken("|<unk>|", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 1: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}


In [5]:
class BPEAdapter:
    name = "bpe-tokenizer_alphabet_nlp_dataset_29,000"

    def __init__(self, path):
        self.path = Path(path)

        if not self.path.is_file():
            raise FileNotFoundError(
                f"Tokenizer file not found: {self.path}"
            )

        self.backend = HFTokenizer.from_file(str(self.path))

        self.unk_id = self.backend.token_to_id("|<unk>|")

        self.pad_id = self.backend.token_to_id("<pad>")

        if self.pad_id is None:
            self.backend.add_special_tokens(["<pad>"])
            self.pad_id = self.backend.token_to_id("<pad>")

        if self.pad_id is None:
            raise RuntimeError("Could not initialize PAD token.")

        self.vocab_size = self.backend.get_vocab_size()

        self.bos_id = self.backend.token_to_id("<bos>")
        self.eos_id = self.backend.token_to_id("<eos>")

        self.backend.enable_padding(
            pad_id=self.pad_id,
            pad_token="<pad>"
        )

        if self.vocab_size <= 0:
            raise ValueError("Tokenizer vocabulary is empty.")

        print(f"Tokenizer: {self.name}")
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"PAD ID: {self.pad_id}")
        print(f"UNK ID: {self.unk_id}")

    def encode(self, text, max_length=MAX_TEXT_LENGTH):
        text = str(text).strip()

        if not text:
            raise ValueError("Input text is empty.")

        encoding = self.backend.encode(text)
        ids = encoding.ids[:max_length]

        if not ids:
            raise ValueError(f"No tokens generated for: {text!r}")

        return ids

    def decode(self, ids):
        return self.backend.decode(
            [int(i) for i in ids],
            skip_special_tokens=True
        )


# ---------------------------------------------------------
# Load project tokenizer
# ---------------------------------------------------------

tokenizer = BPEAdapter(TOKENIZER_PATH)

# Checkpoint compatibility signature
tokenizer_signature = (
    "bpe:"
    + hashlib.sha256(
        TOKENIZER_PATH.read_bytes()
    ).hexdigest()
)


# ---------------------------------------------------------
# Batch encoding with dynamic padding
# ---------------------------------------------------------

def encode_batch(texts):

    if not texts:
        raise ValueError("Empty text batch.")

    encoded = [
        tokenizer.encode(
            text,
            MAX_TEXT_LENGTH
        )
        for text in texts
    ]

    max_length = max(
        len(ids) for ids in encoded
    )

    tokens = torch.full(
        (len(encoded), max_length),
        tokenizer.pad_id,
        dtype=torch.long
    )

    for i, ids in enumerate(encoded):

        tokens[i, :len(ids)] = torch.tensor(
            ids,
            dtype=torch.long
        )

    return tokens


print("=" * 55)
print("ARK BPE TOKENIZER")
print("=" * 55)

print("Name:", tokenizer.name)
print("Path:", tokenizer.path)
print("Vocabulary:", tokenizer.vocab_size)
print("PAD:", tokenizer.pad_id)
print("UNK:", tokenizer.unk_id)
print("BOS:", tokenizer.bos_id)
print("EOS:", tokenizer.eos_id)
print("Max length:", MAX_TEXT_LENGTH)
print("Signature:", tokenizer_signature[:24] + "...")

print("=" * 55)

Tokenizer: bpe-tokenizer_alphabet_nlp_dataset_29,000
Vocabulary size: 29874
PAD ID: 29873
UNK ID: 0
ARK BPE TOKENIZER
Name: bpe-tokenizer_alphabet_nlp_dataset_29,000
Path: F:\pyvenv\ARK\ark-vision\tokenizer\bpe-tokenizer_alphabet_nlp_dataset_29,000.json
Vocabulary: 29874
PAD: 29873
UNK: 0
BOS: None
EOS: None
Max length: 64
Signature: bpe:0e44899f064e7db9996f...


In [6]:
TEST_TEXTS = [
    "dog",
    "cat",
    "a book on the table",
    "a beautiful house near the lake",
    "یک سگ",
    "یک کتاب روی میز",
    "یک خانه زیبا کنار دریاچه",
    "ARK Gamma Mini v3",
]

print("=" * 65)
print("TOKENIZER TEST")
print("=" * 65)

for index, text in enumerate(TEST_TEXTS, start=1):

    encoding = tokenizer.backend.encode(text)

    ids = tokenizer.encode(
        text,
        MAX_TEXT_LENGTH
    )

    tokens = [
        tokenizer.backend.id_to_token(i)
        for i in ids
    ]

    decoded = tokenizer.decode(ids)

    print(f"\nTEST {index}")
    print("-" * 65)

    print("Input:")
    print(repr(text))

    print("\nTokens:")
    print(tokens)

    print("\nToken IDs:")
    print(ids)

    print("\nToken count:")
    print(len(ids))

    print("\nDecoded:")
    print(repr(decoded))

    print("\nRound-trip exact match:")
    print(text == decoded)


# =========================================================
# Batch encoding test
# =========================================================

print("\n" + "=" * 65)
print("BATCH ENCODING TEST")
print("=" * 65)

batch = encode_batch(TEST_TEXTS)

print("Batch shape:", tuple(batch.shape))

print("\nEncoded batch:")
print(batch)

print("\nPadding mask:")
padding_mask = batch.eq(tokenizer.pad_id)

print(padding_mask)

print("\nValid token counts:")
print((~padding_mask).sum(dim=1).tolist())

# =========================================================
# Validation
# =========================================================

assert batch.ndim == 2

assert batch.shape[0] == len(TEST_TEXTS)

assert batch.shape[1] <= MAX_TEXT_LENGTH

assert not padding_mask.all(dim=1).any(), (
    "At least one sequence contains only padding."
)

assert batch.min().item() >= 0

assert batch.max().item() < tokenizer.vocab_size

print("\nAll structural tokenizer tests passed.")


TOKENIZER TEST

TEST 1
-----------------------------------------------------------------
Input:
'dog'

Tokens:
['d', 'og']

Token IDs:
[69, 2046]

Token count:
2

Decoded:
'dog'

Round-trip exact match:
True

TEST 2
-----------------------------------------------------------------
Input:
'cat'

Tokens:
['cat']

Token IDs:
[11789]

Token count:
1

Decoded:
'cat'

Round-trip exact match:
True

TEST 3
-----------------------------------------------------------------
Input:
'a book on the table'

Tokens:
['a', 'Ġbook', 'Ġon', 'Ġthe', 'Ġt', 'able']

Token IDs:
[66, 16961, 2928, 1565, 820, 3816]

Token count:
6

Decoded:
'a book on the table'

Round-trip exact match:
True

TEST 4
-----------------------------------------------------------------
Input:
'a beautiful house near the lake'

Tokens:
['a', 'Ġbe', 'a', 'u', 'ti', 'ful', 'Ġhouse', 'Ġn', 'ear', 'Ġthe', 'Ġl', 'ake']

Token IDs:
[66, 2561, 66, 86, 474, 11806, 24071, 705, 1600, 1565, 1852, 4543]

Token count:
12

Decoded:
'a beautiful ho

## 🗂️ Dataset & Captions

In [7]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
}

process = psutil.Process(os.getpid())


def ram_mb():
    return process.memory_info().rss / (1024 ** 2)


def discover_pairs(split_dir):
    """Find image-caption pairs only in selected dataset folders."""
    pairs = []
    skipped = 0
    folders = set()

    if not split_dir.is_dir():
        return pairs, skipped, folders

    for dataset_name in SELECTED_DATASETS:
        dataset_dir = split_dir / dataset_name

        if not dataset_dir.is_dir():
            print(f"WARNING: Dataset directory not found: {dataset_dir}")
            continue

        for image_path in dataset_dir.rglob("*"):
            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            caption_path = image_path.with_suffix(".txt")

            if not caption_path.is_file():
                skipped += 1
                continue

            try:
                caption = caption_path.read_text(
                    encoding="utf-8-sig"
                ).strip()

                if not caption:
                    skipped += 1
                    continue

                pairs.append((image_path, caption))
                folders.add(image_path.parent)

            except (OSError, UnicodeError):
                skipped += 1

    return pairs, skipped, folders


print("========== DATASET REPORT ==========")
print(f"ROOT: {ROOT}")
print(f"DATA: {DATA}")
print(f"DATA exists: {DATA.exists()}")
print(f"Selected datasets: {SELECTED_DATASETS}")

ram_before = ram_mb()
print(f"RAM before loading: {ram_before:.1f} MB")

reports = {}
total_samples = 0
total_skipped = 0

for split in ("train",):
    split_dir = DATA / split

    pairs, skipped, folders = discover_pairs(split_dir)

    reports[split] = pairs
    total_samples += len(pairs)
    total_skipped += skipped

    ram_now = ram_mb()
    ram_delta = ram_now - ram_before

    print(
        f"\n{split.upper()}: {len(pairs)} image-caption pairs "
        f"| skipped={skipped}"
    )
    print(
        f"Folders: {len(folders)} "
        f"| RAM={ram_now:.1f} MB "
        f"(delta={ram_delta:+.1f} MB)"
    )

    folder_counts = {}

    for image_path, _ in pairs:
        relative_folder = image_path.parent.relative_to(split_dir)
        folder_name = str(relative_folder).replace("\\", "/") + "/"

        folder_counts[folder_name] = (
            folder_counts.get(folder_name, 0) + 1
        )

    for folder_name, count in sorted(folder_counts.items()):
        print(f"  {folder_name}: {count} samples")

    if pairs:
        image_path, caption = random.choice(pairs)
        print("  Random sample:")
        print(f"    Image: {image_path.name}")
        print(f"    Caption: {caption}")
    else:
        print("  No valid image-caption pairs found.")

print("\n========== SUMMARY ==========")
print(f"Selected datasets: {SELECTED_DATASETS}")
print(
    f"Total training samples: {len(reports['train'])}"
)
print(f"Total skipped: {total_skipped}")

ram_end = ram_mb()
print(f"RAM at end: {ram_end:.1f} MB")
print(f"RAM increase: {ram_end - ram_before:+.1f} MB")
print("=============================")

========== DATASET REPORT ==========
ROOT: F:\pyvenv\ARK\ark-vision
DATA: F:\pyvenv\ARK\ark-vision\dataset\Alphabet_VISION_Dataset
DATA exists: True
Selected datasets: ['various_images']
RAM before loading: 293.2 MB

TRAIN: 19 image-caption pairs | skipped=0
Folders: 1 | RAM=293.2 MB (delta=+0.0 MB)
  various_images/: 19 samples
  Random sample:
    Image: doctor_doom_1.png
    Caption: Doctor Doom | Doctor Doom in the film Avengers: Doomsday

========== SUMMARY ==========
Selected datasets: ['various_images']
Total training samples: 19
Total skipped: 0
RAM at end: 293.2 MB
RAM increase: +0.0 MB


In [8]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def discover_pairs(root):
    root = Path(root)

    if not root.is_dir():
        raise FileNotFoundError(
            f"Dataset directory not found: {root}"
        )

    pairs = []
    missing_captions = 0
    empty_captions = 0

    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue

        # Caption must have the same filename stem
        caption_file = p.with_suffix(".txt")

        if not caption_file.is_file():
            missing_captions += 1
            continue

        caption = caption_file.read_text(
            encoding="utf-8-sig"
        ).strip()

        if not caption:
            empty_captions += 1
            continue

        pairs.append((p, caption))

    print(
        f"Scanned: {root.name} | "
        f"Pairs: {len(pairs)} | "
        f"Missing captions: {missing_captions} | "
        f"Empty captions: {empty_captions}"
    )

    return pairs


# ------------------------------------------------------------
# Discover training pairs
# ------------------------------------------------------------

train_pairs = []

for directory in TRAIN_DIRS:
    dataset_pairs = discover_pairs(directory)
    train_pairs.extend(dataset_pairs)


if not train_pairs:
    raise ValueError(
        "No training image-caption pairs found. "
        "Check TRAIN_DIRS and matching .txt caption files."
    )


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class ImageCaptionDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = list(pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, caption = self.pairs[idx]

        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert(IMAGE_MODE)

            # PIL resize order is (width, height)
            im = im.resize(
                (IMAGE_SIZE[1], IMAGE_SIZE[0]),
                Image.Resampling.BICUBIC
            )

            arr = (
                np.asarray(im, dtype=np.float32).copy()
                / 127.5
                - 1.0
            )

        if IMAGE_CHANNELS == 1:
            arr = arr[None, :, :]
        else:
            arr = arr.transpose(2, 0, 1)

        return torch.from_numpy(arr), caption, idx


# ------------------------------------------------------------
# Batch collation
# ------------------------------------------------------------

def collate(batch):
    images, captions, indices = zip(*batch)

    return (
        torch.stack(images),
        encode_batch(captions),
        list(captions),
        torch.tensor(indices, dtype=torch.long)
    )


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

train_dataset = ImageCaptionDataset(train_pairs)

train_loader = DataLoader(
    ImageCaptionDataset(train_pairs),
    batch_size=4,
    shuffle=True,
    collate_fn=collate,
    num_workers=NUM_WORKERS
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

SINGLE_PAIR = len(train_pairs) == 1

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Selected datasets: {SELECTED_DATASETS}")
print(f"Training pairs:    {len(train_pairs)}")
print(f"Single-pair mode:  {SINGLE_PAIR}")
print("=" * 60)

print("\nTraining examples:")
for path, caption in train_pairs[:5]:
    print(f"{path.name} -> {caption!r}")

Scanned: various_images | Pairs: 19 | Missing captions: 0 | Empty captions: 0

DATASET SUMMARY
Selected datasets: ['various_images']
Training pairs:    19
Single-pair mode:  False

Training examples:
asteroid_impact.jpg -> 'Asteroid Impact | The Chicxulub asteroid impact at the Cretaceous-Paleogene boundary'
cheyava_falls.jpg -> 'Cheyava Falls | An image of Cheyava Falls, where the white markings might be traces of ancient microbial activity'
cretaceous.jpg -> 'Cretaceous Period | Dinosaurs roaming a prehistoric landscape'
doctor_doom_1.png -> 'Doctor Doom | Doctor Doom in the film Avengers: Doomsday'
doctor_doom_2.jpg -> 'Doctor Doom | A man with a green cloak and a metal mask'


In [9]:
SINGLE_PAIR = len(train_pairs) == 1

print(f"Train pairs: {len(train_pairs)}")
print(f"Single-pair mode: {SINGLE_PAIR}")

Train pairs: 19
Single-pair mode: False


## 🧠 Model Architecture

In [11]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.embed = nn.Embedding(vocab_size, TEXT_DIM, padding_idx=pad_id)
        self.pos = nn.Embedding(MAX_TEXT_LENGTH, TEXT_DIM)
        layer = nn.TransformerEncoderLayer(TEXT_DIM, NUM_HEADS, TEXT_DIM*4,
                                           dropout=0.1, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.net = nn.TransformerEncoder(layer, NUM_TEXT_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(TEXT_DIM)
    def forward(self, tokens):
        b, t = tokens.shape
        if t > MAX_TEXT_LENGTH: raise ValueError("Sequence too long.")
        mask = tokens.eq(self.pad_id)
        if mask.all(dim=1).any(): raise ValueError("All-padding text sequence.")
        pos = torch.arange(t, device=tokens.device)[None].expand(b,-1)
        x = self.embed(tokens) + self.pos(pos)
        return self.norm(self.net(x, src_key_padding_mask=mask)), mask

class CrossAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.qnorm = nn.LayerNorm(channels)
        self.tnorm = nn.LayerNorm(TEXT_DIM)
        self.attn = nn.MultiheadAttention(channels, NUM_HEADS, kdim=TEXT_DIM,
                                           vdim=TEXT_DIM, batch_first=True)
        self.gain = nn.Parameter(torch.tensor(0.1))
    def forward(self, x, text, mask):
        b,c,h,w = x.shape
        q = x.flatten(2).transpose(1,2)
        a,_ = self.attn(self.qnorm(q), self.tnorm(text), self.tnorm(text),
                        key_padding_mask=mask, need_weights=False)
        return (q + self.gain*a).transpose(1,2).reshape(b,c,h,w)

def up(cin, cout):
    return nn.Sequential(nn.ConvTranspose2d(cin,cout,4,2,1),
                         nn.GroupNorm(8,cout),nn.GELU())

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        b=BASE_CHANNELS
        self.text = TextEncoder(tokenizer.vocab_size, tokenizer.pad_id)
        self.fc = nn.Linear(LATENT_DIM+TEXT_DIM,b*8*4*4)
        self.up1, self.attn1 = up(b*8,b*4), CrossAttention(b*4)
        self.up2, self.attn2 = up(b*4,b*2), CrossAttention(b*2)
        self.up3, self.attn3 = up(b*2,b), CrossAttention(b)
        self.up4 = up(b,b)
        self.out = nn.Sequential(nn.Conv2d(b,b,3,padding=1),nn.GELU(),
                                 nn.Conv2d(b,IMAGE_CHANNELS,3,padding=1),nn.Tanh())
    def forward(self, tokens, z=None, size=IMAGE_SIZE):
        b=tokens.size(0)
        if z is None: z=torch.randn(b,LATENT_DIM,device=tokens.device)
        text,mask=self.text(tokens)
        valid=(~mask).unsqueeze(-1).to(text.dtype)
        pooled=(text*valid).sum(1)/valid.sum(1).clamp_min(1)
        x=self.fc(torch.cat([z,pooled],dim=1)).reshape(b,BASE_CHANNELS*8,4,4)
        x=self.attn1(self.up1(x),text,mask)
        x=self.attn2(self.up2(x),text,mask)
        x=self.attn3(self.up3(x),text,mask)
        x=self.up4(x)
        if x.shape[-2:]!=tuple(size):
            x=F.interpolate(x,size=size,mode="bilinear",align_corners=False)
        return self.out(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        b=BASE_CHANNELS
        self.text=TextEncoder(tokenizer.vocab_size,tokenizer.pad_id)
        self.image=nn.Sequential(
            nn.Conv2d(IMAGE_CHANNELS,b,4,2,1),nn.LeakyReLU(.2),
            nn.Conv2d(b,b*2,4,2,1),nn.GroupNorm(8,b*2),nn.LeakyReLU(.2),
            nn.Conv2d(b*2,b*4,4,2,1),nn.GroupNorm(8,b*4),nn.LeakyReLU(.2),
            nn.Conv2d(b*4,b*4,4,2,1),nn.GroupNorm(8,b*4),nn.LeakyReLU(.2),
            nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.text_proj=nn.Linear(TEXT_DIM,b*4)
        self.head=nn.Sequential(nn.Linear(b*8,b*4),nn.LeakyReLU(.2),nn.Linear(b*4,1))
    def forward(self, images, tokens):
        text,mask=self.text(tokens)
        valid=(~mask).unsqueeze(-1).to(text.dtype)
        pooled=(text*valid).sum(1)/valid.sum(1).clamp_min(1)
        joint=torch.cat([self.image(images),self.text_proj(pooled)],dim=1)
        return self.head(joint).squeeze(1)

G,D=Generator().to(DEVICE),Discriminator().to(DEVICE)


# ============================================================
# MODEL SUMMARY
# ============================================================

trainable_params = sum(
    p.numel()
    for model in (G, D)
    for p in model.parameters()
    if p.requires_grad
)

print(f"\n🚀 Model: {MODEL_NAME}")
print("🧩 Model Type: Conditional Generative Adversarial Network (cGAN)")
print(f"🧠 Initialized on: {DEVICE}")
print(f"🔢 Trainable parameters (G + D): {trainable_params:,}")
print("🏗️ Generator: Transformer Text Encoder + CNN Upsampling + Cross-Attention")
print("🛡️ Discriminator: CNN Image Encoder + Transformer Text Encoder")
print("🔗 Conditioning: Text-Image Joint Representation")


🚀 Model: ark-gamma-mini-v2
🧩 Model Type: Conditional Generative Adversarial Network (cGAN)
🧠 Initialized on: cpu
🔢 Trainable parameters (G + D): 10,544,167
🏗️ Generator: Transformer Text Encoder + CNN Upsampling + Cross-Attention
🛡️ Discriminator: CNN Image Encoder + Transformer Text Encoder
🔗 Conditioning: Text-Image Joint Representation


## 🔴 Loss

In [12]:
bce=nn.BCEWithLogitsLoss()
opt_g=torch.optim.Adam(G.parameters(),lr=LR_G,betas=(0.5,0.999))
opt_d=torch.optim.Adam(D.parameters(),lr=LR_D,betas=(0.5,0.999))

def wrong_caption_tokens(captions):
    # Use a distinct caption from this batch; if unavailable, skip mismatch.
    wrong=[]
    for caption in captions:
        alternatives=[x for x in captions if x!=caption]
        wrong.append(random.choice(alternatives) if alternatives else caption)
    mask=torch.tensor([a!=b for a,b in zip(wrong,captions)],device=DEVICE,dtype=torch.bool)
    return encode_batch(wrong).to(DEVICE),mask

def set_requires_grad(module, value):
    for p in module.parameters(): p.requires_grad_(value)

def display_images(images, captions, cols=3):
    images = images.detach().cpu().float()
    images = (images.clamp(-1, 1) + 1) / 2

    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(5 * cols, 5 * rows),
        squeeze=False
    )

    fig.subplots_adjust(
        left=0.03,
        right=0.97,
        top=0.90,
        bottom=0.04,
        hspace=0.35,
        wspace=0.15
    )

    for i, ax in enumerate(axes.flat):
        ax.axis("off")

        if i >= n:
            continue

        image = images[i].permute(1, 2, 0).numpy()

        if image.shape[-1] == 1:
            ax.imshow(image.squeeze(-1), cmap="gray")
        else:
            ax.imshow(image)

        # Wrap long captions onto multiple lines.
        caption = textwrap.fill(str(captions[i]), width=32)
        ax.set_title(caption, fontsize=11, pad=10, wrap=True)

    return fig

## 🗂️ Run directory, checkpoint compatibility & resume

In [13]:
# ============================================================
# CHECKPOINT CONFIGURATION
# ============================================================

# Include training pairs in the dataset signature.
all_pairs_for_signature = [
    ("train", str(p.resolve()), c) for p, c in train_pairs
]

DATA_SIGNATURE = hashlib.sha256(
    json.dumps(
        all_pairs_for_signature,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

CONFIG_SIGNATURE = {
    "model": MODEL_NAME,
    "tokenizer": tokenizer_signature,
    "vocab": tokenizer.vocab_size,
    "pad": tokenizer.pad_id,
    "channels": IMAGE_CHANNELS,
    "image_size": IMAGE_SIZE,
    "latent": LATENT_DIM,
    "text_dim": TEXT_DIM,
    "base": BASE_CHANNELS,
    "heads": NUM_HEADS,
    "layers": NUM_TEXT_LAYERS,
    "max_text": MAX_TEXT_LENGTH,
    "data": DATA_SIGNATURE,
}


def compatible(ck):
    saved_config = ck.get("config")

    if not isinstance(saved_config, dict):
        return False

    # Require exact configuration and dataset/caption signature.
    for key, current_value in CONFIG_SIGNATURE.items():
        if saved_config.get(key) != current_value:
            return False

    return True


def checkpoint_candidates():
    return sorted(
        LOG_ROOT.glob("*/checkpoints/last.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )


# ============================================================
# FIXED PROMPTS
# ============================================================

fixed_prompts = [
    "Doctor Doom | Doctor Doom in the film Avengers: Doomsday",
    "Iron Man | A man in red nanometal armor",
    "Sauron | Sauron in the film The Lord of the Rings",
]

fixed_tokens = encode_batch(fixed_prompts).to(DEVICE)

_fixed_rng = torch.Generator(device="cpu")
_fixed_rng.manual_seed(SEED)

fixed_z = torch.randn(
    len(fixed_prompts),
    LATENT_DIM,
    generator=_fixed_rng
).to(DEVICE)

if not fixed_prompts:
    raise ValueError("No training captions available.")

fixed_tokens = encode_batch(fixed_prompts).to(DEVICE)

# Fixed noise ensures comparable samples between epochs.
_fixed_rng = torch.Generator(device="cpu")
_fixed_rng.manual_seed(SEED)

fixed_z = torch.randn(
    len(fixed_prompts),
    LATENT_DIM,
    generator=_fixed_rng
).to(DEVICE)


# ============================================================
# TRAIN EPOCH
# ============================================================

def train_epoch():
    G.train()
    D.train()

    d_losses = []
    g_losses = []
    adv_losses = []
    recon_losses = []
    mismatch_losses = []

    for images, tokens, captions, _ in tqdm(
        train_loader,
        desc="Training",
        leave=False
    ):
        images = images.to(DEVICE)
        tokens = tokens.to(DEVICE)

        batch_size = images.size(0)

        real_targets = torch.full(
            (batch_size,),
            1.0 - LABEL_SMOOTHING,
            device=DEVICE
        )
        fake_targets = torch.zeros(batch_size, device=DEVICE)

        # ----------------------------------------------------
        # 1. Train Discriminator
        # ----------------------------------------------------

        set_requires_grad(D, True)
        opt_d.zero_grad(set_to_none=True)

        with torch.no_grad():
            fake_images = G(tokens)

        real_logits = D(images, tokens)
        fake_logits = D(fake_images.detach(), tokens)

        loss_real = bce(real_logits, real_targets)
        loss_fake = bce(fake_logits, fake_targets)

        loss_d = 0.5 * (loss_real + loss_fake)

        wrong_tokens, mismatch_mask = wrong_caption_tokens(captions)
        mismatch_count = int(mismatch_mask.sum().item())

        if mismatch_count > 0 and LAMBDA_MISMATCH > 0:
            wrong_tokens = wrong_tokens.to(DEVICE)

            wrong_logits = D(
                images[mismatch_mask],
                wrong_tokens[mismatch_mask]
            )

            loss_mismatch = bce(
                wrong_logits,
                torch.zeros_like(wrong_logits)
            )

            loss_d = loss_d + LAMBDA_MISMATCH * loss_mismatch
            mismatch_losses.append(loss_mismatch.item())

        loss_d.backward()
        opt_d.step()

        # ----------------------------------------------------
        # 2. Train Generator
        # ----------------------------------------------------

        set_requires_grad(D, False)
        opt_g.zero_grad(set_to_none=True)

        fake_images = G(tokens)
        fake_logits = D(fake_images, tokens)

        loss_adv = bce(fake_logits, torch.ones_like(fake_logits))

        # Reconstruction is useful for a single image-caption pair.
        if SINGLE_PAIR:
            loss_recon = F.l1_loss(fake_images, images)
            loss_g = loss_adv + LAMBDA_RECON_SINGLE * loss_recon
        else:
            loss_recon = torch.zeros((), device=DEVICE)
            loss_g = loss_adv

        loss_g.backward()
        opt_g.step()

        set_requires_grad(D, True)

        d_losses.append(loss_d.item())
        g_losses.append(loss_g.item())
        adv_losses.append(loss_adv.item())
        recon_losses.append(loss_recon.item())

    if not d_losses:
        raise RuntimeError("Training loader produced no batches.")

    return {
        "train_D_loss": float(np.mean(d_losses)),
        "train_G_loss": float(np.mean(g_losses)),
        "train_adv_loss": float(np.mean(adv_losses)),
        "train_recon_loss": float(np.mean(recon_losses)),
        "train_mismatch_loss": (
            float(np.mean(mismatch_losses))
            if mismatch_losses else 0.0
        ),
    }


def format_duration(seconds):
    seconds = max(0, int(seconds))

    days, seconds = divmod(seconds, 86400)
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)

    return (
        f"{days}d {hours:02d}h "
        f"{minutes:02d}m {seconds:02d}s"
    )


print("Checkpoint candidates:")

for candidate in checkpoint_candidates():
    print(candidate)


# ============================================================
# RESUME CHECKPOINT
# ============================================================

ck = None

if RESUME_FROM_PREVIOUS:
    for candidate in checkpoint_candidates():
        try:
            loaded = torch.load(
                candidate,
                map_location=DEVICE,
                weights_only=False
            )

            if compatible(loaded):
                ck = loaded
                print("Resuming from:", candidate)
                break

            print("Skipping incompatible checkpoint:", candidate)

        except Exception as exc:
            print("Skipping checkpoint:", candidate, "| Error:", exc)

if ck is None:
    accumulated_training_seconds = 0.0
    RUN_DIR = LOG_ROOT / datetime.now().strftime(
        "%Y-%m-%d_%H-%M-%S_%f"
    )
    start_epoch = 1
    history = []

else:
    RUN_DIR = Path(ck["run_dir"])

    G.load_state_dict(ck["G"])
    D.load_state_dict(ck["D"])
    opt_g.load_state_dict(ck["opt_g"])
    opt_d.load_state_dict(ck["opt_d"])

    start_epoch = ck["epoch"] + 1
    history = ck["history"]

    accumulated_training_seconds = float(
        ck.get("accumulated_training_seconds", 0.0)
    )

    if "torch_rng" in ck:
        torch.set_rng_state(ck["torch_rng"].cpu())

    if "numpy_rng" in ck:
        np.random.set_state(ck["numpy_rng"])

    if "python_rng" in ck:
        random.setstate(ck["python_rng"])

    if DEVICE.type == "cuda" and ck.get("cuda_rng") is not None:
        torch.cuda.set_rng_state_all(ck["cuda_rng"])


# ============================================================
# RUN DIRECTORIES
# ============================================================

CKPT_DIR = RUN_DIR / "checkpoints"
SAMPLE_DIR = RUN_DIR / "samples"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_LOG = RUN_DIR / "training.txt"


# ============================================================
# PERSISTENT LOGGING
# ============================================================

def write_training_log(message):
    """Print a timestamped message and append it to training.txt."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"

    print(line)

    with TRAINING_LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()


# ============================================================
# CHECKPOINT SAVING
# ============================================================

def save_checkpoint(path, epoch):
    payload = {
        "config": CONFIG_SIGNATURE,
        "run_dir": str(RUN_DIR),
        "epoch": epoch,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "opt_g": opt_g.state_dict(),
        "opt_d": opt_d.state_dict(),
        "history": history,
        "accumulated_training_seconds": accumulated_training_seconds,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if DEVICE.type == "cuda"
            else None
        ),
    }

    temp = path.with_suffix(".tmp")
    torch.save(payload, temp)
    temp.replace(path)


# ============================================================
# LOG SESSION START
# ============================================================

write_training_log("=" * 65)
write_training_log(f"Model: {MODEL_NAME}")
write_training_log(f"Run directory: {RUN_DIR}")
write_training_log(f"Device: {DEVICE}")
write_training_log(f"Image mode: {IMAGE_MODE}")
write_training_log(f"Image channels: {IMAGE_CHANNELS}")
write_training_log(f"Training pairs: {len(train_pairs)}")
write_training_log(f"Single-pair mode: {SINGLE_PAIR}")
write_training_log(f"Resume enabled: {RESUME_FROM_PREVIOUS}")
write_training_log(f"Starting epoch: {start_epoch}")
write_training_log(f"Planned final epoch: {start_epoch + EPOCHS - 1}")
write_training_log(f"Checkpoint directory: {CKPT_DIR}")
write_training_log(f"Sample directory: {SAMPLE_DIR}")
write_training_log(f"Training log: {TRAINING_LOG}")
write_training_log(
    f"Accumulated training time before this session: "
    f"{format_duration(accumulated_training_seconds)}"
)
write_training_log("=" * 65)

print("Run:", RUN_DIR, "| epoch range:",
      start_epoch, "-", start_epoch + EPOCHS - 1)

Checkpoint candidates:
[2026-10-31 15:39:20] =================================================================
[2026-10-31 15:39:20] Model: ark-gamma-mini-v2
[2026-10-31 15:39:20] Run directory: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772
[2026-10-31 15:39:20] Device: cpu
[2026-10-31 15:39:20] Image mode: RGB
[2026-10-31 15:39:20] Image channels: 3
[2026-10-31 15:39:20] Training pairs: 19
[2026-10-31 15:39:20] Single-pair mode: False
[2026-10-31 15:39:20] Resume enabled: True
[2026-10-31 15:39:20] Starting epoch: 1
[2026-10-31 15:39:20] Planned final epoch: 50
[2026-10-31 15:39:20] Checkpoint directory: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints
[2026-10-31 15:39:20] Sample directory: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples
[2026-10-31 15:39:20] Training log: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\training.txt
[2026-10-31 15:39:20] Ac

## 〽️ Training

In [14]:
write_training_log("Training loop started.")

try:
    for epoch in range(start_epoch, start_epoch + EPOCHS):
        tic = time.perf_counter()

        write_training_log(f"Epoch {epoch} started.")

        metrics = train_epoch()

        epoch_seconds = time.perf_counter() - tic
        accumulated_training_seconds += epoch_seconds

        row = {
            "epoch": epoch,
            **metrics,
            "epoch_seconds": round(epoch_seconds, 2),
            "total_training_seconds": round(
                accumulated_training_seconds, 2
            ),
            "total_training_time": format_duration(
                accumulated_training_seconds
            ),
        }

        history.append(row)

        write_training_log(
            "METRICS " + json.dumps(
                row,
                ensure_ascii=False,
                default=str
            )
        )

        write_training_log(
            f"Epoch time: {format_duration(epoch_seconds)} | "
            f"Total training time: "
            f"{format_duration(accumulated_training_seconds)}"
        )

        # Generate fixed-prompt samples.
        G.eval()

        with torch.no_grad():
            samples = G(
                fixed_tokens,
                z=fixed_z,
                size=IMAGE_SIZE
            )

        sample_path = SAMPLE_DIR / "latest.png"

        fig = display_images(samples, fixed_prompts)
        try:
            fig.savefig(
                sample_path,
                dpi=120,
                bbox_inches="tight"
            )
        finally:
            plt.close(fig)

        write_training_log(f"Sample saved: {sample_path}")

        # Save the latest checkpoint after every epoch.
        last_checkpoint_path = CKPT_DIR / "last.pt"
        save_checkpoint(last_checkpoint_path, epoch)
        
        write_training_log(
            f"Latest checkpoint saved: {last_checkpoint_path} "
            f"| epoch={epoch} "
            f"| G_loss={metrics['train_G_loss']:.6f}"
        )
        
        # Persist complete history.
        history_path = RUN_DIR / "history.json"
        history_path.write_text(
            json.dumps(
                history,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        write_training_log(
            f"Epoch {epoch} completed | "
            f"Epoch time: {format_duration(epoch_seconds)} | "
            f"Total training time: "
            f"{format_duration(accumulated_training_seconds)}"
        )

    write_training_log(
        f"Training session finished successfully | "
        f"Total training time: "
        f"{format_duration(accumulated_training_seconds)}"
    )

except Exception as exc:
    write_training_log(
        f"TRAINING ERROR: {type(exc).__name__}: {exc}"
    )
    raise

finally:
    write_training_log("Training loop exited.")

[2026-10-31 15:39:24] Training loop started.
[2026-10-31 15:39:24] Epoch 1 started.


[2026-10-31 15:39:34] METRICS {"epoch": 1, "train_D_loss": 0.9781758069992066, "train_G_loss": 0.973249101638794, "train_adv_loss": 0.973249101638794, "train_recon_loss": 0.0, "train_mismatch_loss": 0.5828004479408264, "epoch_seconds": 9.83, "total_training_seconds": 9.83, "total_training_time": "0d 00h 00m 09s"}
[2026-10-31 15:39:34] Epoch time: 0d 00h 00m 09s | Total training time: 0d 00h 00m 09s
[2026-10-31 15:39:37] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:39:41] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=1 | G_loss=0.973249
[2026-10-31 15:39:41] Epoch 1 completed | Epoch time: 0d 00h 00m 09s | Total training time: 0d 00h 00m 09s
[2026-10-31 15:39:41] Epoch 2 started.


[2026-10-31 15:39:53] METRICS {"epoch": 2, "train_D_loss": 0.8840205669403076, "train_G_loss": 1.2831277132034302, "train_adv_loss": 1.2831277132034302, "train_recon_loss": 0.0, "train_mismatch_loss": 0.5547954738140106, "epoch_seconds": 12.42, "total_training_seconds": 22.25, "total_training_time": "0d 00h 00m 22s"}
[2026-10-31 15:39:53] Epoch time: 0d 00h 00m 12s | Total training time: 0d 00h 00m 22s
[2026-10-31 15:39:54] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:40:04] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=2 | G_loss=1.283128
[2026-10-31 15:40:04] Epoch 2 completed | Epoch time: 0d 00h 00m 12s | Total training time: 0d 00h 00m 22s
[2026-10-31 15:40:04] Epoch 3 started.


[2026-10-31 15:40:12] METRICS {"epoch": 3, "train_D_loss": 0.8642685770988464, "train_G_loss": 1.4971042156219483, "train_adv_loss": 1.4971042156219483, "train_recon_loss": 0.0, "train_mismatch_loss": 0.512672072649002, "epoch_seconds": 7.91, "total_training_seconds": 30.16, "total_training_time": "0d 00h 00m 30s"}
[2026-10-31 15:40:12] Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 00m 30s
[2026-10-31 15:40:13] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:40:20] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=3 | G_loss=1.497104
[2026-10-31 15:40:20] Epoch 3 completed | Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 00m 30s
[2026-10-31 15:40:20] Epoch 4 started.


[2026-10-31 15:40:26] METRICS {"epoch": 4, "train_D_loss": 0.8342127084732056, "train_G_loss": 1.6513022184371948, "train_adv_loss": 1.6513022184371948, "train_recon_loss": 0.0, "train_mismatch_loss": 0.5443442583084106, "epoch_seconds": 5.41, "total_training_seconds": 35.57, "total_training_time": "0d 00h 00m 35s"}
[2026-10-31 15:40:26] Epoch time: 0d 00h 00m 05s | Total training time: 0d 00h 00m 35s
[2026-10-31 15:40:27] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:40:30] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=4 | G_loss=1.651302
[2026-10-31 15:40:30] Epoch 4 completed | Epoch time: 0d 00h 00m 05s | Total training time: 0d 00h 00m 35s
[2026-10-31 15:40:30] Epoch 5 started.


...

[2026-10-31 15:50:45] METRICS {"epoch": 46, "train_D_loss": 0.8314679741859436, "train_G_loss": 1.3538577556610107, "train_adv_loss": 1.3538577556610107, "train_recon_loss": 0.0, "train_mismatch_loss": 0.4338268995285034, "epoch_seconds": 6.35, "total_training_seconds": 374.94, "total_training_time": "0d 00h 06m 14s"}
[2026-10-31 15:50:45] Epoch time: 0d 00h 00m 06s | Total training time: 0d 00h 06m 14s
[2026-10-31 15:50:55] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:50:57] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=46 | G_loss=1.353858
[2026-10-31 15:50:57] Epoch 46 completed | Epoch time: 0d 00h 00m 06s | Total training time: 0d 00h 06m 14s
[2026-10-31 15:50:57] Epoch 47 started.


[2026-10-31 15:51:03] METRICS {"epoch": 47, "train_D_loss": 0.7857755303382874, "train_G_loss": 1.448232364654541, "train_adv_loss": 1.448232364654541, "train_recon_loss": 0.0, "train_mismatch_loss": 0.3221184551715851, "epoch_seconds": 6.78, "total_training_seconds": 381.73, "total_training_time": "0d 00h 06m 21s"}
[2026-10-31 15:51:03] Epoch time: 0d 00h 00m 06s | Total training time: 0d 00h 06m 21s
[2026-10-31 15:51:04] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:51:05] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=47 | G_loss=1.448232
[2026-10-31 15:51:05] Epoch 47 completed | Epoch time: 0d 00h 00m 06s | Total training time: 0d 00h 06m 21s
[2026-10-31 15:51:05] Epoch 48 started.


[2026-10-31 15:51:13] METRICS {"epoch": 48, "train_D_loss": 0.7686965942382813, "train_G_loss": 1.396592950820923, "train_adv_loss": 1.396592950820923, "train_recon_loss": 0.0, "train_mismatch_loss": 0.3444337725639343, "epoch_seconds": 7.28, "total_training_seconds": 389.01, "total_training_time": "0d 00h 06m 29s"}
[2026-10-31 15:51:13] Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 29s
[2026-10-31 15:51:14] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:51:16] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=48 | G_loss=1.396593
[2026-10-31 15:51:16] Epoch 48 completed | Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 29s
[2026-10-31 15:51:16] Epoch 49 started.


[2026-10-31 15:51:23] METRICS {"epoch": 49, "train_D_loss": 0.8024859547615051, "train_G_loss": 1.5074623346328735, "train_adv_loss": 1.5074623346328735, "train_recon_loss": 0.0, "train_mismatch_loss": 0.43335159718990324, "epoch_seconds": 7.17, "total_training_seconds": 396.18, "total_training_time": "0d 00h 06m 36s"}
[2026-10-31 15:51:23] Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 36s
[2026-10-31 15:51:24] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:51:26] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=49 | G_loss=1.507462
[2026-10-31 15:51:26] Epoch 49 completed | Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 36s
[2026-10-31 15:51:26] Epoch 50 started.


[2026-10-31 15:51:34] METRICS {"epoch": 50, "train_D_loss": 0.7794677734375, "train_G_loss": 1.5725849866867065, "train_adv_loss": 1.5725849866867065, "train_recon_loss": 0.0, "train_mismatch_loss": 0.42361499071121217, "epoch_seconds": 7.97, "total_training_seconds": 404.14, "total_training_time": "0d 00h 06m 44s"}
[2026-10-31 15:51:34] Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 44s
[2026-10-31 15:51:35] Sample saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\samples\latest.png
[2026-10-31 15:51:38] Latest checkpoint saved: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-31_15-39-20_012772\checkpoints\last.pt | epoch=50 | G_loss=1.572585
[2026-10-31 15:51:38] Epoch 50 completed | Epoch time: 0d 00h 00m 07s | Total training time: 0d 00h 06m 44s
[2026-10-31 15:51:38] Training session finished successfully | Total training time: 0d 00h 06m 44s
[2026-10-31 15:51:38] Training loop exited.


## 🎨 Generate

In [32]:
last_path = CKPT_DIR / "last.pt"

if not last_path.exists():
    raise FileNotFoundError(
        f"Latest checkpoint not found: {last_path}"
    )

last_ck = torch.load(
    last_path,
    map_location=DEVICE,
    weights_only=False
)

if not compatible(last_ck):
    raise RuntimeError("Latest checkpoint is incompatible.")

G.load_state_dict(last_ck["G"])
G.eval()

# Use exactly the same prompts and noise as latest.png.
with torch.inference_mode():
    samples = G(
        fixed_tokens,
        z=fixed_z,
        size=IMAGE_SIZE
    )

fig = display_images(samples, fixed_prompts)

try:
    output_path = SAMPLE_DIR / "generation_samples_latest.png"
    fig.savefig(output_path, dpi=120, bbox_inches="tight")

    print("Checkpoint epoch:", last_ck["epoch"])
    print("Loaded checkpoint:", last_path)
    print("Saved generated samples:", output_path)

finally:
    plt.close(fig)

Checkpoint epoch: 100
Loaded checkpoint: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-30_19-07-22_082808\checkpoints\last.pt
Saved generated samples: F:\pyvenv\ARK\ark-vision\logs\ark-gamma-mini-v2\2026-10-30_19-07-22_082808\samples\generation_samples_latest.png
